<a href="https://colab.research.google.com/github/cristiangromero/LGRT-PLP/blob/main/Actividad3_PasswordValidator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import re
import ipywidgets as widgets
from IPython.display import display

# Función de evaluación---------------------------------------------------------

def evaluate_password(password: str) -> dict:
    """
    Evalúa la contraseña y devuelve:
    estado, progreso (0-100), estilo visual y criterios pendientes.
    """
    # Campo vacío
    if len(password) == 0:
        return {"status": "Esperando...", "progress": 0,
                "style": "info", "missing": []}

    # Caso: la contraseña contiene espacios (no permitidos)
    if ' ' in password:
        return {"status": "ERROR", "progress": 0,
                "style": "danger", "missing": ["No se permiten espacios"]}

    # Contraseña por debajo del mínimo aceptable
    if len(password) < 6:
        return {"status": "Muy corta", "progress": 25,
                "style": "danger", "missing": ["Mínimo 6 caracteres"]}

    # Criterios de fortaleza
    criteria = {
        "Mínimo 10 caracteres": len(password) >= 10,
        "Al menos una mayúscula": bool(re.search(r'[A-Z]', password)),
        "Al menos una minúscula": bool(re.search(r'[a-z]', password)),
        "Al menos un número": bool(re.search(r'\d', password)),
        "Al menos un caracter especial": bool(re.search(r'[^A-Za-z0-9]', password)),
    }

    score = sum(criteria.values()) # criterios cumplidos
    missing = [k for k, v in criteria.items() if not v] # criterios pendientes

    if score <= 2:
        return {"status": "Débil",     "progress": 50,  "style": "danger",  "missing": missing}
    elif score <= 4:
        return {"status": "Aceptable", "progress": 75,  "style": "warning", "missing": missing}
    else:
        return {"status": "Fuerte",  "progress": 100, "style": "success", "missing": []}


# Interfaz de usuario ----------------------------------------------------------

password_input = widgets.Password(
    description = "Contraseña:",
    placeholder = "Ingrese su contraseña..."
)
progress_bar = widgets.IntProgress(
    value = 0, min = 0, max = 100, bar_style = "info",
    layout = widgets.Layout(width = "300px")
)
status_label = widgets.Label(value = "Esperando...")
criteria_label = widgets.Label(value = "")

# Manejador de eventos ---------------------------------------------------------

def on_password_change(change):
    """
    Se ejecuta automáticamente cada vez que el usuario modifica la contraseña.
    'change' contiene el valor anterior y el nuevo.
    """
    result = evaluate_password(change['new']) # evalúa el nuevo valor

    # Actualiza los widgets con el resultado de la evaluación
    status_label.value = f"Estado: {result['status']}"
    progress_bar.value = result['progress']
    progress_bar.bar_style = result['style']

    # Muestra criterios pendientes o confirmación de éxito
    if result['missing']:
        criteria_label.value = "Pendiente: " + " | ".join(result['missing'])
    elif result['progress'] == 100:
        criteria_label.value = "Todos los criterios cumplidos"
    else:
        criteria_label.value = ""

# Registra el manejador para que se ejecute ante cada cambio del campo
password_input.observe(on_password_change, names ='value')

# Muestra la interfaz en el entorno Jupyter / Colab
display(password_input, progress_bar, status_label, criteria_label)

Password(description='Contraseña:', placeholder='Ingrese su contraseña...')

IntProgress(value=0, bar_style='info', layout=Layout(width='300px'))

Label(value='Esperando...')

Label(value='')